# Etape 0 : Connexion Elasticsearch

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

print(es.info())

{'name': 'ahmed-HP-EliteBook-840-G1', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'W4B7ErSDS-G_v4p6lxl0sg', 'version': {'number': '9.3.2', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '43a703737aab6baefa748bc7b69e4054926f2b2c', 'build_date': '2026-03-16T13:12:56.143057855Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


# 1 Créer un index avec mapping

In [3]:
index_name = "products_v1"

mapping = {
    "mappings": {
        "properties": {
            "name": {"type": "text"},
            "category": {"type": "keyword"},
            "price": {"type": "float"},
            "created_at": {"type": "date"}
        }
    }
}

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

# 2 Ajouter des documents

In [4]:
docs = [
    {"id": 1, "name": "iphone 15 Pro", "category": "smartphone", "price": 1200},
    {"id": 2, "name": "Samsung Galaxy", "category": "smartphone", "price": 900},
    {"id": 3, "name": "Dell Laptop", "category": "laptop", "price": 1500},
]

for doc in docs:
    es.index(index=index_name, id=doc["id"], document=doc)

print("Documents insérés")

Documents insérés


# 3 Vérifier les données

In [7]:
res = es.search(index=index_name, query={"match_all": {}})

for hit in res["hits"]["hits"]:
    print(hit['_source'])

{'id': 1, 'name': 'iphone 15 Pro', 'category': 'smartphone', 'price': 1200}
{'id': 2, 'name': 'Samsung Galaxy', 'category': 'smartphone', 'price': 900}
{'id': 3, 'name': 'Dell Laptop', 'category': 'laptop', 'price': 1500}


# 4 Tester TEXT vs KEYWORD

In [8]:
res = es.search(index=index_name, query={
    "match": {"name": "iphone"}
})
print("Match: ", [h["_source"] for h in res["hits"]["hits"]])

res = es.search(index=index_name, query={
    "term": {"category": "smartphone"}
})
print("Term : ", [h["_source"] for h in res["hits"]["hits"]])

Match:  [{'id': 1, 'name': 'iphone 15 Pro', 'category': 'smartphone', 'price': 1200}]
Term :  [{'id': 1, 'name': 'iphone 15 Pro', 'category': 'smartphone', 'price': 1200}, {'id': 2, 'name': 'Samsung Galaxy', 'category': 'smartphone', 'price': 900}]


# 5 Ajouter un alias (Production)

In [9]:
alias_name = "products"

try:
    es.indices.delete_alias(index="_all", name=alias_name)
except:
    pass

es.indices.put_alias(index=index_name, name=alias_name)

print("alias créé")

alias créé


In [10]:
es.search(index="products", query={"match_all": {}})

ObjectApiResponse({'took': 0, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products_v1', '_id': '1', '_score': 1.0, '_source': {'id': 1, 'name': 'iphone 15 Pro', 'category': 'smartphone', 'price': 1200}}, {'_index': 'products_v1', '_id': '2', '_score': 1.0, '_source': {'id': 2, 'name': 'Samsung Galaxy', 'category': 'smartphone', 'price': 900}}, {'_index': 'products_v1', '_id': '3', '_score': 1.0, '_source': {'id': 3, 'name': 'Dell Laptop', 'category': 'laptop', 'price': 1500}}]}})

# 6 Reindex (migration)

In [15]:
new_index = "products_v2"

mapping_v2 = {
    "mappings": {
        "properties": {
            "name": {"type": "text"},
            "category": {"type": "keyword"},
            "price": {"type": "float"},
            "discount": {"type": "float"}
        }
    }
}

if es.indices.exists(index=new_index):
    es.indices.delete(index=new_index)

es.indices.create(index=new_index, body=mapping_v2)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'products_v2'})

# 7 Switch alias (Zero Downtime)

In [16]:
actions = [
    {"remove": {"index": index_name, "alias": alias_name}},
    {"add": {"index": new_index, "alias": alias_name}}
]

es.indices.update_aliases(body={"actions": actions})

print("Alias switch vers v2")

Alias switch vers v2


# 8 index template (automatisation)

In [32]:
template = {
    "index_patterns": ["logs-*"],
    "template": {
        "settings": {
            "number_of_shards": 1
        },

        "mappings": {
            "properties": {
                "message": {"type": "text"},
                "level": {"type": "keyword"}
            }
        }
    }
}

es.indices.put_index_template(name="logs_template", body=template)
print("Template créé")

Template créé


In [34]:
es.index(index="logs-2026-04-02", document={
    "message": "error occurred",
    "level": "ERROR"
})

mapping = es.indices.get_mapping(index="logs-2026-04-02")
print(mapping)

{'.ds-logs-2026-04-02-2026.04.02-000001': {'mappings': {'_data_stream_timestamp': {'enabled': True}, 'dynamic_templates': [{'ecs_timestamp': {'match': '@timestamp', 'mapping': {'ignore_malformed': False, 'type': 'date'}}}, {'ecs_message_match_only_text': {'path_match': ['message', '*.message'], 'unmatch_mapping_type': 'object', 'mapping': {'type': 'match_only_text'}}}, {'ecs_non_indexed_keyword': {'path_match': ['*event.original', '*gen_ai.agent.description'], 'mapping': {'doc_values': False, 'index': False, 'type': 'keyword'}}}, {'ecs_non_indexed_long': {'path_match': '*.x509.public_key_exponent', 'mapping': {'doc_values': False, 'index': False, 'type': 'long'}}}, {'ecs_ip': {'path_match': ['ip', '*.ip', '*_ip'], 'match_mapping_type': 'string', 'mapping': {'type': 'ip'}}}, {'ecs_wildcard': {'path_match': ['*.io.text', '*.message_id', '*registry.data.strings', '*url.path'], 'unmatch_mapping_type': 'object', 'mapping': {'type': 'wildcard'}}}, {'ecs_path_match_wildcard_and_match_only_tex

In [31]:
from elasticsearch import NotFoundError

index_name = "logs-2026-04-02"

try:
    es.indices.delete(index=index_name)
    print(f"Succès : L'index '{index_name}' a été supprimé.")
except NotFoundError:
    print(f"Note : L'index '{index_name}' n'existait déjà plus. Rien à faire.")
except Exception as e:
    print(f"Erreur inattendue : {e}")

Note : L'index 'logs-2026-04-02' n'existait déjà plus. Rien à faire.
